# Montandon EDA — coverage, linkage, and features



## Setup

In [ ]:
from monty_tool.api_utils import get_pystac_client, get_collection_items
import os

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))


In [ ]:
import re
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch

# Fixed categorical order — a series keeps its colour no matter which chart it's in
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
INK, INK_MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#e1e0d9", "#c3c2b7"
# Single-hue ramp for magnitude (heatmaps): light = low, dark = high
BLUES = LinearSegmentedColormap.from_list("blues", ["#cde2fb", "#3987e5", "#0d366b"])

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK_MUTED,
    "xtick.color": "#898781", "ytick.color": "#898781",
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "legend.fontsize": 8,
})


def hbar(labels, values, title, color=COLORS[0], fmt="{:,.0f}", ax=None):
    """Horizontal bars, first label at the top, value printed at the end of each bar."""
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 0.32 * len(labels) + 1))
    bars = ax.barh(labels[::-1], values[::-1], color=color[::-1] if isinstance(color, list) else color, height=0.7)
    ax.bar_label(bars, labels=[fmt.format(v) for v in values[::-1]], padding=3, fontsize=8, color=INK_MUTED)
    ax.set_title(title)
    ax.grid(axis="y", visible=False)
    ax.margins(x=0.15)
    return ax

Smoke test through the validated path on the newest items.

In [ ]:
# Start small — max_items caps how many you pull
items = get_collection_items("emdat-events", max_items=20)

print(len(items))
first = items[0]
print(first.id, first.collection)
print(first.properties.title)
print(first.properties.monty_country_codes)
print(first.properties.start_datetime, first.properties.end_datetime)


## Collection census

The API doesn't report `numberMatched` on any endpoint, so `get_collection_counts()` pages every collection at 1,000 IDs per page (STAC `fields` extension strips everything else) and sums `numberReturned`. Hours on the staging server, so it's cached to `data/collection_counts.json`. `null` = uncounted: `usgs-events` passed 1M IDs after 6.5 h of paging and was cancelled; USGS hazards/impacts, PDC hazards/impacts, and DesInventar were never reached. All seven were already in the skip/sample tier.

Notable: every source has `events == hazards` exactly (1:1) except IFRC, which has **0** hazards. `ibtracs-hazards` is 722,837 — one item per storm-track observation, ~54 per storm. `cems-*`, `reference-events`, and `rsh-test2-events` are empty.

In [ ]:
import json
from pathlib import Path

from monty_tool.api_utils import get_collection_counts

# data/ is gitignored, so this cache is local-only
counts_path = Path("../data/collection_counts.json")

if counts_path.exists():
    counts = json.loads(counts_path.read_text())
else:
    counts = get_collection_counts()
    counts_path.parent.mkdir(exist_ok=True)
    counts_path.write_text(json.dumps(counts, indent=2))

# keys starting with "_" are file metadata, not collections
rows = {k: v for k, v in counts.items() if not k.startswith("_")}
for cid, n in sorted(rows.items(), key=lambda kv: -(kv[1] or -1)):
    print(f"{n:>9,}  {cid}" if n is not None else f"{'(uncounted)':>9}  {cid}")
print(f"{sum(n for n in rows.values() if n):>9,}  TOTAL counted")

In [ ]:
# Counted, non-empty collections only. ibtracs-hazards is one item per storm-track fix, so it dwarfs the rest.
census = sorted(((n, cid) for cid, n in rows.items() if n), reverse=True)
hbar([cid for _, cid in census], [n for n, _ in census], "Items per collection")
plt.show()

## Pulling the core collections

The core collections for all three tracks: EM-DAT (canonical disaster records + impacts, 1900–present), GDACS (second source with alert levels; the natural overlap test), and IFRC's own events/impacts (tiny, but the only operational-response data). ~137k items, 48 MB gzipped.

EM-DAT is pulled **without geometry**: its items carry full-resolution country MultiPolygons (mean 33 KB, max 10.8 MB for `emdat-event-2002-0495-CHL`), >99% of the payload, and a 1,000-item page times out the server's ~30 s gateway. `fields={"exclude": ["geometry"]}` is honoured server-side and cuts the pull from ~1.5 GB / 90 min to 10 MB / 4 min. `bbox` is kept; `country_codes` + any boundary set reproduces the polygon.

In [ ]:
from pystac_client import Client

from monty_tool.api_utils import STAC_API_URL, _get_headers
from monty_tool.data_cache import load_collection, pull_collection, raw_cache_path

# EM-DAT geometry is >99% of its payload (full-res country polygons, up to 10 MB per Item)
# and duplicates monty:country_codes, so it's pulled without geometry.
TIER1 = {
    "emdat-events": False,
    "emdat-impacts": False,
    "gdacs-events": True,
    "ifrcevent-events": True,
    "ifrcevent-impacts": True,
}
CACHE_DIR = Path("../data/raw")

# The staging server silently drops connections under load; without a timeout a pull can hang forever
client = Client.open(STAC_API_URL, headers=_get_headers(), timeout=60)

for cid, geom in TIER1.items():
    if not raw_cache_path(cid, CACHE_DIR, geometry=geom).exists():
        pull_collection(cid, cache_dir=CACHE_DIR, client=client, geometry=geom)

raw = {cid: list(load_collection(cid, CACHE_DIR, geometry=geom)) for cid, geom in TIER1.items()}
for cid, items in raw.items():
    print(f"{len(items):>7,}  {cid}")

## Flatten to Polars

`items_to_frame` promotes `properties` to columns under their API names (`monty:corr_id`, not `monty_corr_id`) and flattens nested dicts with dotted keys (`monty:impact_detail.value`). Polars, to match the `embedding-setup` branch. Column null-rates *are* field coverage.

In [ ]:
import polars as pl
from collections import Counter
from pydantic import ValidationError

from monty_tool.api_schemas import MontandonItem
from monty_tool.data_cache import items_to_frame

# Wider display defaults so the coverage / linkage tables below aren't truncated
pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(30)
pl.Config.set_fmt_str_lengths(70)

# One frame per collection, then a single tall frame. diagonal_relaxed fills columns a
# collection lacks (e.g. impact_detail.* on events) with null — those nulls are the coverage signal.
frames = {cid: items_to_frame(items) for cid, items in raw.items()}
df = pl.concat(frames.values(), how="diagonal_relaxed")

# Each collection keeps the same colour in every chart
COLL_COLOR = dict(zip(frames, COLORS))
df.shape

In [ ]:
df.head(20)

## 1. Field coverage

% of items with a non-null value, per field, per collection. **Every key the API returns is here** — including ones `MontandonItem` doesn't model (`monty:etl_id`, `monty:guid`, `processing:*`), which `extra="ignore"` drops. Anything we want from those has to come through the raw path.

What to look for:
- `keywords` 1–31%, `monty:src_event_id` 0–21% — see section 2 for why.
- `monty:impact_detail.unit` is 100% on EM-DAT impacts and **0%** on IFRC impacts.
- `severitydata` is a GDACS-only field; it shows up on 31% of GDACS *events*.
- `geometry_type` 0% on EM-DAT is our pull (`geometry=False`), not the API.

In [ ]:
# Identity / bookkeeping columns that are trivially 100%
SKIP = {"id", "collection", "bbox", "n_links", "n_assets"}
coverage = (
    df.group_by("collection")
    # % non-null per column, per collection
    .agg([(pl.col(c).is_not_null().mean() * 100).round(0).alias(c) for c in df.columns if c not in SKIP])
    # long form (collection, field, pct) then re-pivot so fields are rows and collections are columns
    .unpivot(index="collection", variable_name="field", value_name="pct")
    .pivot(on="collection", index="field", values="pct")
    .sort("field")
)
coverage

In [ ]:
# Same table as a heatmap: dark = present, light = missing. The pale rows are the schema's required fields.
fields = coverage["field"].to_list()
cols = sorted(c for c in coverage.columns if c != "field")
mat = coverage.select(cols).to_numpy()

fig, ax = plt.subplots(figsize=(7, 0.3 * len(fields) + 1))
im = ax.imshow(mat, cmap=BLUES, vmin=0, vmax=100, aspect="auto")
ax.set_xticks(range(len(cols)), cols, rotation=30, ha="right")
ax.set_yticks(range(len(fields)), fields)
ax.grid(False)
# print the number in each cell so colour never carries the value alone
for i in range(len(fields)):
    for j in range(len(cols)):
        v = mat[i, j]
        ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=7, color="white" if v > 55 else INK)
ax.set_title("Field coverage (% of items non-null)")
plt.show()

## 2. ETL generations

`keywords` and `monty:src_event_id` aren't missing by *collection* — they're missing by *when the ETL ran*. Items dated 2025+ have `keywords`; items dated 2026 also have `monty:src_event_id` and `processing:version = 0.2.4` (`pystac-monty`). Everything older — ~97% of the bank — predates those fields. Same shape in every collection.

The ETL changelog (v1.0.0, 2026-06-30) describes "separate historical and latest pipelines": history was backfilled by an older code path. **Open question for IFRC:** will the historical pipeline be re-run on pystac-monty 0.2.4? Until it is, `keywords` is only usable on the ~3% of items that have it — or reconstructed from `hazard_codes` + `country_codes`, which is all the ETL does anyway.

In [ ]:
# Presence of the three "new ETL" fields, bucketed by the item's event year (a proxy for when it was loaded)
generations = (
    df.with_columns(pl.col("datetime").str.slice(0, 4).alias("year"))
    .group_by("collection", "year")
    .agg(
        pl.len().alias("n"),
        (pl.col("keywords").is_not_null().mean() * 100).round(0).alias("keywords_pct"),
        (pl.col("monty:src_event_id").is_not_null().mean() * 100).round(0).alias("src_event_id_pct"),
        (pl.col("processing:version").is_not_null().mean() * 100).round(0).alias("proc_version_pct"),
    )
    .filter(pl.col("year") >= "2022")  # everything earlier is 0% across the board
    .sort("collection", "year")
)
# Only one version string exists in the whole bank, so proc_version_pct == "% produced by pystac-monty 0.2.4"
print("processing:version values:", df["processing:version"].drop_nulls().unique().to_list())
generations

In [ ]:
# One panel per field, one bar per collection per year. The step up in 2025/2026 is the same in every collection.
years = sorted(generations["year"].unique().to_list())
width = 0.15

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
for ax, col in zip(axes, ["keywords_pct", "src_event_id_pct"]):
    for k, cid in enumerate(frames):
        sub = generations.filter(pl.col("collection") == cid)
        pct = dict(zip(sub["year"], sub[col]))
        xs = [i + (k - 2) * (width + 0.01) for i in range(len(years))]  # offset each collection within the year group
        ax.bar(xs, [pct.get(y, 0) for y in years], width=width, color=COLL_COLOR[cid], label=cid)
    ax.set_xticks(range(len(years)), years)
    ax.set_title(f"{col.removesuffix('_pct')} present (% of items)")
    ax.grid(axis="x", visible=False)
axes[0].legend(loc="upper left")
plt.show()

## 3. Schema validation

Sanity check that the validated path (`MontandonItem.model_validate`) accepts the whole cache and that impact Items
land on the impact properties model — `extra="ignore"` would otherwise drop `monty:impact_detail` silently. EM-DAT
gets a bbox-centre Point standing in for its excluded geometry, so this tests the schema, not our pull.

In [ ]:
def with_geometry(item: dict) -> dict:
    """Stand in a bbox-centre Point where geometry was excluded from the pull."""
    if item.get("geometry") is None and item.get("bbox"):
        b = item["bbox"]
        return {**item, "geometry": {"type": "Point", "coordinates": [(b[0] + b[2]) / 2, (b[1] + b[3]) / 2]}}
    return item

rows = []
for cid, items in raw.items():
    models: Counter[str] = Counter()
    invalid = 0
    for it in items:
        try:
            models[type(MontandonItem.model_validate(with_geometry(it)).properties).__name__] += 1
        except ValidationError:
            invalid += 1
    rows.append({"collection": cid, "n": len(items), "invalid": invalid, "properties_model": " / ".join(sorted(models))})
pl.DataFrame(rows)

## 4. Text fields

What's actually in `title` and `description`? This decides which collection is a usable NLP / embedding corpus.

- **EM-DAT:** `Flood in Lushoto, Korogwe districts (Tanga region), United Republic of Tanzania of February 1993` — formulaic `<Hazard> in <Place>, <Country> of <Month Year>`. The only content not already in `hazard_codes` / `country_codes` / `datetime` is the sub-national place name.
- **GDACS:** `Green M 4.6 Earthquake in Indonesia at: 28 Apr 2005 04:48:13.` — fully templated. Embeddings will cluster by template, not by event.
- **IFRC:** `NA` for 85% of events; the rest is raw HTML from a rich-text editor. The ~300 real ones are long (p90 1.3k chars, max 79k) and are the only free-form narrative among the five collections pulled. Needs HTML stripping before any NLP.

Implication for the embedding track: `keywords` is embeddable on <1% of items, `description` is templated on the two big collections. Title/description similarity is still useful as a *matching feature* across sources; it's weak as a clustering target.

In [ ]:
# Length distribution plus two placeholder detectors: the literal "NA" and anything that looks like an HTML tag
text_stats = (
    df.with_columns(
        pl.col("description").str.len_chars().alias("desc_len"),
        pl.col("title").str.len_chars().alias("title_len"),
    )
    .group_by("collection")
    .agg(
        pl.len().alias("n"),
        pl.col("desc_len").median().alias("desc_median"),
        pl.col("desc_len").quantile(0.9).alias("desc_p90"),
        pl.col("desc_len").max().alias("desc_max"),
        pl.col("description").n_unique().alias("desc_unique"),  # low vs n => templated / repeated text
        (pl.col("description") == "NA").sum().alias("desc_is_NA"),
        pl.col("description").str.contains("<[a-z]+").sum().alias("desc_has_html"),
        pl.col("title_len").median().alias("title_median"),
    )
    .sort("collection")
)
text_stats

In [ ]:
# What each collection's `description` actually holds. NA and HTML never overlap, so the three parts sum to 100.
comp = text_stats.select(
    "collection",
    (pl.col("desc_is_NA") / pl.col("n") * 100).alias("literal 'NA'"),
    (pl.col("desc_has_html") / pl.col("n") * 100).alias("raw HTML"),
).with_columns((100 - pl.col("literal 'NA'") - pl.col("raw HTML")).alias("plain text"))

fig, ax = plt.subplots(figsize=(7, 2.6))
left = [0.0] * comp.height
for name, color in [("plain text", COLORS[0]), ("raw HTML", COLORS[1]), ("literal 'NA'", AXIS)]:
    vals = comp[name].to_list()
    ax.barh(comp["collection"].to_list(), vals, left=left, color=color, label=name, height=0.6, edgecolor="white", linewidth=1.5)
    left = [l + v for l, v in zip(left, vals)]
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.grid(axis="y", visible=False)
ax.set_title("What `description` contains (% of items)")
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.15))
plt.show()

In [ ]:
for cid in raw:
    print(f"--- {cid} ---")
    for d in frames[cid].sample(4, seed=1)["description"].to_list():
        print("  ", d[:160].replace("\n", " "))

In [ ]:
# The IFRC "description" field is a placeholder 85% of the time
frames["ifrcevent-events"].group_by("description").len().sort("len", descending=True).head(5)

## 5. `monty:corr_id` linkage

`corr_id` is the documented cross-source correlation key. How well does it actually link?

- **Within-source:** every EM-DAT impact and every IFRC impact joins to an event on `corr_id` (100%).
- **Cross-source:** only 470 of 56,712 event `corr_id`s (0.8%) appear in more than one source collection.

In [ ]:
# For every corr_id: which collections it appears in, and how many items share it
corr = df.group_by("monty:corr_id").agg(
    pl.col("collection").unique().sort().alias("collections"),
    pl.len().alias("n_items"),
)
# Roll up by the *set* of collections — e.g. ["emdat-events", "emdat-impacts"] is the within-source link,
# anything mixing emdat / gdacs / ifrcevent is a cross-source link
linkage = corr.group_by("collections").agg(
    pl.len().alias("n_corr_ids"),
    pl.col("n_items").sum().alias("n_items"),
).sort("n_corr_ids", descending=True)

# Cross-source rate, counted on event collections only (impacts always share their own event's corr_id)
events = df.filter(pl.col("collection").str.ends_with("-events"))
n_sources = events.group_by("monty:corr_id").agg(pl.col("collection").n_unique().alias("n_sources"))
multi = n_sources.filter(pl.col("n_sources") > 1).height
print(f"unique corr_ids: {corr.height:,}")
print(f"event corr_ids seen in >1 source: {multi:,} of {n_sources.height:,} ({100 * multi / n_sources.height:.1f}%)")

# Within-source rate: what share of impacts semi-join to an event on corr_id (GDACS has no impacts pulled)
for src in ["emdat", "ifrcevent"]:
    ev, im = frames[f"{src}-events"], frames[f"{src}-impacts"]
    joined = im.join(ev.select("monty:corr_id").unique(), on="monty:corr_id", how="semi").height
    print(f"{src}-impacts -> {src}-events join: {100 * joined / im.height:.1f}%")
linkage

In [ ]:
# corr_ids by the set of collections they appear in. Orange = the set spans more than one source.
# Log x-axis: the cross-source sets are two orders of magnitude smaller and vanish on a linear scale.
top = linkage.head(10)
sets = top["collections"].to_list()
cross = [len({c.split("-")[0] for c in s}) > 1 for s in sets]
ax = hbar(
    [" + ".join(s) for s in sets],
    top["n_corr_ids"].to_list(),
    "corr_ids by collection set (top 10, log scale)",
    color=[COLORS[1] if x else COLORS[0] for x in cross],
)
ax.set_xscale("log")
ax.set_xlim(1, 2e5)
ax.legend(handles=[Patch(color=COLORS[0], label="single source"), Patch(color=COLORS[1], label="cross-source")], loc="lower right")
plt.show()

### Why it can't work across sources

From the ETL repo (`pystac_monty/paring.py`), `corr_id` is deterministic:
`{YYYYMMDD}-{country}-{geoblock}-{hazard_cluster}-{episode}-GCDB`. Three format generations coexist in the bank, and **GDACS puts its own event ID in the `episode` slot** (`…-GH0101-1732529-GCDB`) where EM-DAT and IFRC put `1`. So a GDACS and an EM-DAT record of the same event can never share a `corr_id`, in any format. The geoblock in the current format makes matching stricter still — the current-format IDs have a **0.0%** cross-source rate.

IFRC's own cookbook (recipe 07, Türkiye–Syria 2023) doesn't use `corr_id` across sources either: it links by `bbox` + `datetime` window + a hand-enumerated list of hazard codes across every vocabulary. That's the approach to replicate, with the crosswalk replacing the hand-typed lists.

In [ ]:
# corr_id format generations. Current format has a numeric geoblock between country and cluster:
#   current: 20260828-ZWE-638247-TL0405-1-GCDB   older: 20251216-IDN-MH0600-1-GCDB
CURRENT = r"^\d{8}-[A-Z]{3}-\d+-[A-Z]{2}\d{4}-\d+-GCDB$"
ev_fmt = (
    df.filter(pl.col("collection").str.ends_with("-events"))
    .with_columns(
        pl.col("datetime").str.slice(0, 4).alias("year"),
        pl.col("monty:corr_id").str.contains(CURRENT).alias("current_format"),
    )
)
# How many events per collection are on each format, with an example ID of each
print(ev_fmt.group_by("collection", "current_format").agg(pl.len().alias("n"), pl.col("monty:corr_id").first().alias("example")).sort("collection", "current_format"))
print()

# Cross-source rate split by format — does the stricter current format link *better* or *worse*?
by_fmt = ev_fmt.group_by("monty:corr_id").agg(pl.col("collection").n_unique().alias("n_src"), pl.col("current_format").first())
by_fmt.group_by("current_format").agg(
    pl.len().alias("corr_ids"),
    (pl.col("n_src") > 1).sum().alias("multi_source"),
    ((pl.col("n_src") > 1).mean() * 100).round(2).alias("multi_source_pct"),
)

## 6. Categoricals and geometry

- **Hazard codes:** most items carry one code, but ~27k carry 3 or 5 — one per taxonomy. Top values mix EM-DAT (`nat-hyd-flo-riv`), UNDRR-ISC (`GH0001`, `EN0013`), and GLIDE (`FL`, `WF`) vocabularies in the same field. Section 7 crosswalks them to a `cluster_label` / `family_label` via the ETL repo's `HazardProfiles.csv`.
- **Countries:** 98% single-country.
- **Geometry:** GDACS = Point; IFRC = Polygon/MultiPolygon; EM-DAT = country MultiPolygons (excluded here).
- **Date ranges:** EM-DAT 1900–2026; GDACS 2000–2026; IFRC 1970–2026.

In [ ]:
# hazard_codes is a list column: 1 code = one vocabulary; 3 or 5 = the same hazard tagged in every vocabulary
print("hazard codes per item:")
print(df["monty:hazard_codes"].list.len().value_counts().sort("monty:hazard_codes"))
print("\ntop hazard codes (note the mixed vocabularies — EM-DAT `nat-hyd-flo-riv`, UNDRR `GH0001`, GLIDE `FL`):")
# explode so each code counts once per item it appears on
df.explode("monty:hazard_codes", empty_as_null=True).group_by("monty:hazard_codes").len().sort("len", descending=True).head(15)

In [ ]:
# Top hazard codes, coloured by which vocabulary the code belongs to
def vocab(code: str) -> str:
    if re.fullmatch(r"[A-Z]{2}\d{4}", code):
        return "UNDRR-ISC"
    if re.fullmatch(r"[A-Z]{2}", code):
        return "GLIDE"
    if re.fullmatch(r"[a-z]{3}(-[a-z]{3})+", code):
        return "EM-DAT"
    return "other"

VOCAB_COLOR = {"EM-DAT": COLORS[0], "UNDRR-ISC": COLORS[1], "GLIDE": COLORS[2], "other": AXIS}
top_codes = df.explode("monty:hazard_codes", empty_as_null=True).group_by("monty:hazard_codes").len().sort("len", descending=True).head(15)
codes = top_codes["monty:hazard_codes"].to_list()
ax = hbar(codes, top_codes["len"].to_list(), "Top 15 hazard codes", color=[VOCAB_COLOR[vocab(c)] for c in codes])
ax.legend(handles=[Patch(color=c, label=v) for v, c in VOCAB_COLOR.items() if v != "other"], loc="lower right")
plt.show()

# and how the whole field splits by vocabulary
vocab_counts = (
    df.explode("monty:hazard_codes", empty_as_null=True)
    .with_columns(pl.col("monty:hazard_codes").map_elements(vocab, return_dtype=pl.String).alias("vocab"))
    .group_by("vocab").len().sort("len", descending=True)
)
vocab_counts

In [ ]:
# Same shape as hazard codes: how many countries per item, then the most-tagged countries
print("countries per item:")
print(df["monty:country_codes"].list.len().value_counts().sort("monty:country_codes"))
print("\ntop countries:")
df.explode("monty:country_codes", empty_as_null=True).group_by("monty:country_codes").len().sort("len", descending=True).head(10)

In [ ]:
top_countries = df.explode("monty:country_codes", empty_as_null=True).group_by("monty:country_codes").len().sort("len", descending=True).head(15)
hbar(top_countries["monty:country_codes"].to_list(), top_countries["len"].to_list(), "Top 15 countries (items tagged)")
plt.show()

In [ ]:
# Geometry types (EM-DAT is null because we pulled it with geometry=False)
print(df.group_by("collection", "geometry_type").len().sort("collection"))
# Date span per collection
print(df.group_by("collection").agg(
    pl.col("datetime").min().str.slice(0, 10).alias("min_date"),
    pl.col("datetime").max().str.slice(0, 10).alias("max_date"),
).sort("collection"))
# What EM-DAT impact records actually measure
frames["emdat-impacts"].group_by("monty:impact_detail.category", "monty:impact_detail.type").len().sort("len", descending=True)

In [ ]:
# Events per year, one panel per collection (own y-scale each — GDACS's spikes would flatten the others).
# EM-DAT is steady from ~1990; GDACS is patchy (2004–05, 2022, 2026 only), consistent with the staging DB still loading.
per_year = (
    df.filter(pl.col("collection").str.ends_with("-events"))
    .with_columns(pl.col("datetime").str.slice(0, 4).cast(pl.Int32).alias("year"))
    .group_by("collection", "year").len()
    .sort("collection", "year")
)
event_colls = per_year["collection"].unique().sort().to_list()
fig, axes = plt.subplots(len(event_colls), 1, figsize=(9, 2.2 * len(event_colls)), sharex=True)
for ax, cid in zip(axes, event_colls):
    sub = per_year.filter(pl.col("collection") == cid)
    ax.plot(sub["year"], sub["len"], color=COLL_COLOR[cid], linewidth=2)
    ax.set_title(f"{cid} per year")
axes[-1].set_xlim(1950, 2027)  # pre-1950 EM-DAT is a handful of items per year
plt.show()

# What EM-DAT impacts actually measure
impact_types = frames["emdat-impacts"].group_by("monty:impact_detail.type").len().sort("len", descending=True)
hbar(impact_types["monty:impact_detail.type"].to_list(), impact_types["len"].to_list(), "EM-DAT impact records by type")
plt.show()

## 7. Feature engineering

`monty_tool.features` turns the flattened frame into one row per event with derived columns: hazard cluster/family
from the vendored crosswalk, start/end/duration, bbox centroid and area, impacts pivoted per `corr_id`, and a cleaned
description. Every function only adds columns, so `df`'s coverage numbers above still apply.

This section checks whether those features are usable: does the crosswalk recognise the codes the bank actually uses,
are the derived time/space values sane per source, how many events actually get impacts joined, and what text survives
cleaning.

In [ ]:
import numpy as np

from monty_tool.features import (
    IMPACT_TYPES, add_clean_description, event_feature_table, hazard_code_lookup,
)

# Events get every feature; impacts are pivoted to one row per corr_id and left-joined on
events = df.filter(pl.col("collection").str.ends_with("-events"))
impacts = df.filter(pl.col("collection").str.ends_with("-impacts"))
feat = add_clean_description(event_feature_table(events, impacts))
event_colls = [c for c in COLL_COLOR if c.endswith("-events")]

new_cols = [c for c in feat.columns if c not in df.columns]
print(f"{feat.height:,} events x {feat.width} columns; {len(new_cols)} derived:")
print(new_cols)
feat.select(
    "id", "collection", "hazard_code_primary", "hazard_cluster", "year", "duration_days",
    "centroid_lat", "centroid_lon", "death", "affected_total",
).head(8)

### Hazard crosswalk

`add_hazard_features` walks each item's `monty:hazard_codes` in order and takes the first code the crosswalk knows,
after fixing known typos (`nat-gem-ear-gro`). Coverage is the share of events that got a cluster at all.

In [ ]:
lookup = hazard_code_lookup()
print(f"crosswalk knows {lookup.height} distinct codes across {lookup['hazard_cluster'].n_unique()} clusters / {lookup['hazard_family'].n_unique()} families")
print(feat.group_by("collection").agg(
    pl.len().alias("n"),
    (pl.col("hazard_cluster").is_not_null().mean() * 100).round(1).alias("matched_pct"),
    pl.col("n_hazard_codes").mean().round(2).alias("codes_per_item"),
).sort("collection"))

# Every code on the events that fell through. Three codes account for all of it: two EM-DAT
# technological codes and one UNDRR key that HazardProfiles.csv simply lacks.
unmatched = (
    feat.filter(pl.col("hazard_cluster").is_null())
    .explode("monty:hazard_codes", empty_as_null=True)
    .group_by("collection", "monty:hazard_codes").len()
    .sort("len", descending=True)
)
unmatched

In [ ]:
# Hazard family per collection. GDACS is dominated by "Environmental" — that's its ~24k wildfire alerts,
# which the UNDRR taxonomy files under Environmental Degradation. Not a crosswalk error, but it means
# wildfires are the majority class in GDACS and will dominate anything trained on it unstratified.
fam = (
    feat.with_columns(pl.col("hazard_family").fill_null("(unmatched)"))
    .group_by("hazard_family", "collection").len()
    .pivot(on="collection", index="hazard_family", values="len")
    .fill_null(0)
)
fam = fam.with_columns(pl.sum_horizontal(pl.exclude("hazard_family")).alias("total")).sort("total", descending=True)
families = fam["hazard_family"].to_list()

fig, ax = plt.subplots(figsize=(8, 0.5 * len(families) + 1))
left = np.zeros(len(families))
for cid in event_colls:
    vals = fam[cid].to_numpy().astype(float)
    ax.barh(families[::-1], vals[::-1], left=left[::-1], color=COLL_COLOR[cid], label=cid, height=0.7)
    left += vals
ax.set_title("Events by hazard family, stacked by collection")
ax.grid(axis="y", visible=False)
ax.legend(loc="lower right")
plt.show()

# One level finer
clusters = feat.group_by("hazard_cluster").len().sort("len", descending=True).head(15)
hbar(clusters["hazard_cluster"].fill_null("(unmatched)").to_list(), clusters["len"].to_list(), "Top 15 hazard clusters")
plt.show()

# Which code actually got matched, by vocabulary — GDACS items list UNDRR first, so it wins there
matched_vocab = (
    feat.filter(pl.col("hazard_code_primary").is_not_null())
    .with_columns(pl.col("hazard_code_primary").map_elements(vocab, return_dtype=pl.String).alias("vocab"))
    .group_by("collection", "vocab").len()
    .pivot(on="vocab", index="collection", values="len").fill_null(0).sort("collection")
)
matched_vocab

In [ ]:
# EM-DAT composition over time: events per decade, stacked by family. The record is thin before 1960 and
# the growth after is largely reporting, not hazard. Technological's share peaks in the 2000s and falls back.
# The 2020s bar is a partial decade.
decades = (
    feat.filter((pl.col("collection") == "emdat-events") & pl.col("hazard_family").is_not_null())
    .with_columns((pl.col("year") // 10 * 10).alias("decade"))
    .group_by("decade", "hazard_family").len()
    .pivot(on="hazard_family", index="decade", values="len").fill_null(0)
    .sort("decade")
)
fam_order = [f for f in families if f in decades.columns]
fig, ax = plt.subplots(figsize=(9, 3.6))
bottom = np.zeros(decades.height)
palette = COLORS + ["#7a7a7a", "#b8b8b8", "#d9d9d9"]
for color, fam_name in zip(palette, fam_order):
    vals = decades[fam_name].to_numpy().astype(float)
    ax.bar(decades["decade"], vals, bottom=bottom, width=8, color=color, label=fam_name)
    bottom += vals
ax.set_title("EM-DAT events per decade, by hazard family")
ax.grid(axis="x", visible=False)
ax.legend(loc="upper left", ncol=2)
plt.show()

### Time

`duration_days = end_datetime - start_datetime`. Two things to know before using it: IFRC sets `end == start` on
every item, so duration is identically zero there; and EM-DAT has a few hundred events with `end < start`.

In [ ]:
print(feat.group_by("collection").agg(
    pl.col("year").min().alias("min_year"),
    pl.col("year").max().alias("max_year"),
    (pl.col("duration_days") == 0).mean().mul(100).round(0).alias("zero_pct"),
    (pl.col("duration_days") < 0).sum().alias("negative"),
    pl.col("duration_days").median().alias("median"),
    pl.col("duration_days").quantile(0.9).alias("p90"),
    pl.col("duration_days").max().alias("max"),
).sort("collection"))

# Positive durations only, log-x. GDACS (episode-based) is centred on ~5 days; EM-DAT's tail is droughts.
fig, ax = plt.subplots(figsize=(8, 3.2))
bins = np.logspace(0, 3.6, 40)
for cid in event_colls:
    d = feat.filter((pl.col("collection") == cid) & (pl.col("duration_days") > 0))["duration_days"].to_numpy()
    if len(d):
        ax.hist(d, bins=bins, histtype="step", linewidth=1.8, color=COLL_COLOR[cid], label=f"{cid} (n={len(d):,})")
ax.set_xscale("log")
ax.set_xlabel("duration_days (events with duration > 0)")
ax.set_title("Event duration")
ax.legend()
plt.show()

In [ ]:
# Duration is really a property of the hazard: GDACS median and p90 duration per cluster (clusters with 100+ events).
# Droughts run for months, earthquakes are instantaneous, floods/cyclones sit in between.
dur_cluster = (
    feat.filter((pl.col("collection") == "gdacs-events") & pl.col("hazard_cluster").is_not_null())
    .group_by("hazard_cluster").agg(
        pl.len().alias("n"),
        pl.col("duration_days").median().alias("median"),
        pl.col("duration_days").quantile(0.9).alias("p90"),
    )
    .filter(pl.col("n") >= 100)
    .sort("median", descending=True)
)
labels = [f"{c} (n={n:,})" for c, n in zip(dur_cluster["hazard_cluster"], dur_cluster["n"])]
fig, ax = plt.subplots(figsize=(8, 0.45 * len(labels) + 1))
y = np.arange(len(labels))[::-1]
p90_bars = ax.barh(y, dur_cluster["p90"], color=GRID, height=0.7, label="p90")
ax.barh(y, dur_cluster["median"], color=COLL_COLOR["gdacs-events"], height=0.7, label="median")
ax.bar_label(p90_bars, labels=[f"{m:.0f} / {p:.0f}" for m, p in zip(dur_cluster["median"], dur_cluster["p90"])], padding=4, fontsize=8, color=INK_MUTED)
ax.set_yticks(y)
ax.set_yticklabels(labels)
ax.set_xscale("symlog", linthresh=1)  # earthquakes are 0 days; droughts 1,000+
ax.set_xlim(0, 5000)
ax.set_xlabel("duration_days (median / p90)")
ax.set_title("GDACS event duration by hazard cluster")
ax.grid(axis="y", visible=False)
ax.legend(loc="lower right")
plt.show()

In [ ]:
# Seasonality: share of each family's EM-DAT events by start month (rows sum to 1). EM-DAT is the longest,
# steadiest record so it's the only source where this is meaningful. Families under 100 events are dropped —
# Extraterrestrial is one meteor and would otherwise own the colour scale.
season = (
    feat.filter((pl.col("collection") == "emdat-events") & pl.col("hazard_family").is_not_null())
    .filter(pl.len().over("hazard_family") >= 100)
    .group_by("hazard_family", "month").len()
    .pivot(on="month", index="hazard_family", values="len").fill_null(0)
    .sort("hazard_family")
)
month_cols = [pl.col(str(m)) if str(m) in season.columns else pl.lit(0).alias(str(m)) for m in range(1, 13)]
season_np = season.select(month_cols).to_numpy().astype(float)
season_np /= season_np.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(8, 0.45 * season.height + 1))
ax.imshow(season_np, cmap=BLUES, aspect="auto")
ax.set_yticks(range(season.height))
ax.set_yticklabels(season["hazard_family"].to_list())
ax.set_xticks(range(12))
ax.set_xticklabels(list("JFMAMJJASOND"))
ax.grid(False)
ax.set_title("EM-DAT: share of each family's events by start month")
plt.show()

### Space

Everything here is bbox-derived, so it inherits each source's geometry resolution. GDACS Items are Points — their bbox
has zero area and the centroid *is* the location. EM-DAT bboxes are country or admin-region extents (median ~100 deg²), so the
centroid is a region's centre, not the event's. Cross-source spatial matching on bbox alone therefore can't work;
that's what IFRC GO's per-event geometry (`go_api.py`) is for.

In [ ]:
print(feat.group_by("collection").agg(
    (pl.col("bbox_area_deg2") == 0).mean().mul(100).round(0).alias("zero_area_pct"),
    pl.col("bbox_area_deg2").median().round(1).alias("area_median"),
    pl.col("bbox_area_deg2").quantile(0.9).round(1).alias("area_p90"),
    pl.col("n_countries").mean().round(2).alias("countries_mean"),
    (pl.col("n_countries") > 1).sum().alias("multi_country"),
).sort("collection"))

fig, axes = plt.subplots(1, len(event_colls), figsize=(4.4 * len(event_colls), 3.2), sharex=True, sharey=True)
for ax, cid in zip(axes, event_colls):
    sub = feat.filter(pl.col("collection") == cid)
    ax.scatter(sub["centroid_lon"], sub["centroid_lat"], s=2, alpha=0.25, color=COLL_COLOR[cid], linewidths=0)
    ax.set_title(f"{cid} centroids")
    ax.set_xlim(-180, 180)
    ax.set_ylim(-60, 85)
    ax.set_aspect("equal")
plt.show()

In [ ]:
print(feat.group_by("collection").agg(
    pl.len().alias("n"),
    (pl.col("n_impact_records").is_not_null().mean() * 100).round(1).alias("any_impact_pct"),
    *[(pl.col(t).is_not_null().mean() * 100).round(1).alias(t) for t in IMPACT_TYPES if t in feat.columns],
).sort("collection"))

# Impact types the pivot ignores — all IFRC-only measures. Worth adding to IMPACT_TYPES if IFRC matters.
all_types = impacts.group_by("collection", "monty:impact_detail.type").len()
print(all_types.filter(~pl.col("monty:impact_detail.type").is_in(IMPACT_TYPES)).sort("len", descending=True))

# Deaths vs people affected on EM-DAT, top five families — the two headline severity measures
sub = feat.filter((pl.col("collection") == "emdat-events") & (pl.col("death") > 0) & (pl.col("affected_total") > 0) & pl.col("hazard_family").is_not_null())
top_fam = sub.group_by("hazard_family").len().sort("len", descending=True).head(5)["hazard_family"].to_list()
fig, ax = plt.subplots(figsize=(7, 5))
for color, fam_name in zip(COLORS, top_fam):
    s = sub.filter(pl.col("hazard_family") == fam_name)
    ax.scatter(s["affected_total"], s["death"], s=6, alpha=0.4, color=color, linewidths=0, label=f"{fam_name} (n={s.height:,})")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("affected_total")
ax.set_ylabel("death")
ax.set_title(f"EM-DAT events with both measures (n={sub.height:,})")
ax.legend(loc="upper left")
plt.show()

In [ ]:
# Which impact measures each collection actually carries: share of events with a non-null value per measure
measures = [t for t in IMPACT_TYPES if t in feat.columns]
imp_cov = feat.group_by("collection").agg(
    *[(pl.col(t).is_not_null().mean() * 100).alias(t) for t in measures]
).sort("collection")
fig, ax = plt.subplots(figsize=(9, 3.4))
x = np.arange(len(measures))
w = 0.8 / len(event_colls)
for k, cid in enumerate(event_colls):
    vals = imp_cov.filter(pl.col("collection") == cid).select(measures).row(0)
    bars = ax.bar(x + (k - (len(event_colls) - 1) / 2) * w, vals, width=w, color=COLL_COLOR[cid], label=cid)
    ax.bar_label(bars, labels=[f"{v:.0f}" if v >= 1 else "" for v in vals], padding=2, fontsize=7, color=INK_MUTED)
ax.set_xticks(x)
ax.set_xticklabels(measures)
ax.set_ylabel("% of events with the measure")
ax.set_title("Impact measure coverage per collection")
ax.grid(axis="x", visible=False)
ax.legend()
plt.show()

# EM-DAT deaths per year since 1980, stacked by family. Spikes are single events (1983 Ethiopian famine,
# 2004 tsunami, 2010 Haiti, 2023 Türkiye) — the series is its tail, which matters for any severity model.
deaths = (
    feat.filter((pl.col("collection") == "emdat-events") & (pl.col("year") >= 1980) & pl.col("hazard_family").is_not_null())
    .group_by("year", "hazard_family").agg(pl.col("death").sum())
    .pivot(on="hazard_family", index="year", values="death").fill_null(0)
    .sort("year")
)
fig, ax = plt.subplots(figsize=(9, 3.6))
bottom = np.zeros(deaths.height)
for color, fam_name in zip(palette, [f for f in fam_order if f in deaths.columns]):
    vals = deaths[fam_name].to_numpy().astype(float)
    ax.bar(deaths["year"], vals, bottom=bottom, width=0.8, color=color, label=fam_name)
    bottom += vals
ax.set_title("EM-DAT recorded deaths per year, by hazard family")
ax.grid(axis="x", visible=False)
ax.legend(loc="upper right", ncol=2)
plt.show()

### Text

`add_clean_description` strips HTML, collapses whitespace, and nulls `NA`/blank placeholders. On IFRC that nulls ~85%
of descriptions — the placeholders from section 4 — leaving only the long narrative ones.

In [ ]:
print(feat.group_by("collection").agg(
    pl.col("description").str.len_chars().median().alias("raw_median_len"),
    pl.col("description_len").median().alias("clean_median_len"),
    (pl.col("description_clean").is_null() & pl.col("description").is_not_null()).sum().alias("nulled_by_clean"),
    (pl.col("description_clean").is_null().mean() * 100).round(1).alias("null_pct"),
).sort("collection"))

for cid in event_colls:
    sample = feat.filter((pl.col("collection") == cid) & pl.col("description_clean").is_not_null()).sample(2, seed=3)
    for d in sample["description_clean"].to_list():
        print(f"  [{cid}] {d[:150]}")

In [ ]:
# Length of the cleaned description, log-x, each bar is that collection's share so IFRC's 322 are visible. EM-DAT and GDACS are tight template-length peaks; IFRC's surviving
# 15% is an order of magnitude longer and the only distribution that looks like free text.
fig, ax = plt.subplots(figsize=(8, 3.2))
bins = np.logspace(1, 4.5, 45)
for cid in event_colls:
    d = feat.filter(pl.col("collection") == cid)["description_len"].drop_nulls().to_numpy()
    ax.hist(d, bins=bins, weights=np.full(len(d), 1 / len(d)), histtype="step", linewidth=1.8, color=COLL_COLOR[cid], label=f"{cid} (n={len(d):,})")
ax.set_xscale("log")
ax.set_xlabel("description_len (characters, after cleaning)")
ax.set_ylabel("share of collection")
ax.set_title("Description length (each collection normalised)")
ax.legend()
plt.show()

## What the upstream repos tell us

Two IFRC repos explain most of the above. 

**[`IFRCGo/montandon-etl`](https://github.com/IFRCGo/montandon-etl)** (+ `pystac-monty` submodule) — the ETL that produces every item.
- `keywords = hazard_profiles.get_keywords(hazard_codes) + country_codes` in every source. Derived, not sourced.
- `paring.py` — the `corr_id` construction above.
- `HazardProfiles.csv` — the hazard-code crosswalk. `geo_blocks-0.2.parquet` — the spatial grid.
- `extension.py` — canonical field list and enums (`MontyRoles`, `MontyImpactType`, `MontyResponseType`, …) and a `monty:response_detail` object we haven't seen because no response collection has data yet. Schema URI `monty-stac-extension/v1.3.0`.

**[`IFRCGo/montandon-notebooks`](https://github.com/IFRCGo/montandon-notebooks)** ([cookbook](https://ifrcgo.org/montandon-notebooks/)) — nine recipes.
- Recipe 07 links across sources by date + bbox + hazard code, not `corr_id`.
- Recipe 08: the API supports **CQL2 filtering server-side** — `a_overlaps` on `monty:hazard_codes`, `a_contains` on `monty:country_codes`, `t_intersects` on `datetime`, comparisons on `monty:impact_detail.type/.value`. `/queryables` lists filterable fields. Targeted questions don't need a full pull. Filter by *collection*, not `roles` (server bug).
- Per-source field mapping docs at `monty-stac-extension/model/sources/<SOURCE>`, linked from each collection's `describedby`.
